In [1]:
pip install torch torchvision transformers pandas pill easyocr scikit-learn

ERROR: Could not find a version that satisfies the requirement pill (from versions: none)
ERROR: No matching distribution found for pill
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import re
from collections import Counter



In [3]:
# ==========================================
# POLIMEME DECODE v3.0: Text-Only Classification
# (Pure NLP with Dual-Context Processing)
# ==========================================
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
import numpy as np
from tqdm import tqdm
import warnings
import gc

warnings.filterwarnings('ignore')

# ================= CONFIGURATION =================
TRAINING_CONFIG = {
    # Data paths
    'train_csv': '/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Train/Train.csv',
    'test_csv': '/kaggle/input/poli-meme-decode-cuet-cse-fest/PoliMemeDecode/Test/Test.csv',
    'output_dir': '/kaggle/working/outputs',
    
    # Enhanced context paths
    'train_context_csv': '/kaggle/input/cleaned1/cleaned meme data cleaned (1).csv',
    'test_context_csv': '/kaggle/input/cleaned-text/cleaned_meme_text_data.csv',
    
  # Switched from 'Large' (300M+ params) to 'Base' (~85-125M params)
    'primary_model': 'microsoft/deberta-v3-base',   # ~86M params, still SOTA for NLU
    'secondary_model': 'roberta-base',              # ~125M params, faster and lighter
    # Model Strategy Parameters
    'use_dual_encoder': True,  # Use two different models for diversity
    'context_weight': 3.0,
    'ocr_weight': 1.5,
    'ensemble_strategy': 'weighted_avg',  # 'weighted_avg', 'concat', 'attention'
    
    # Training Hyperparameters
    'num_folds': 5,
    'epochs': 10,
    'batch_size': 4,
    'learning_rate': 8e-6,
    'weight_decay': 0.01,
    'max_len_primary': 256,  # Longer sequences for rich context
    'max_len_secondary': 192,
    'num_classes': 2,
    'gradient_accumulation_steps': 8,
    
    # Memory optimization
    'use_gradient_checkpointing': False,
    'mixed_precision': True,
    'empty_cache_freq': 50,
    
    # Device
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
}

# Label Mappings
LABEL2ID = {'NonPolitical': 0, 'Political': 1} 
ID2LABEL = {0: 'NonPolitical', 1: 'Political'}

# ==========================================
# LOAD AND MERGE DATA
# ==========================================
def load_and_prepare_data():
    """Load and merge data with OCR and context"""
    
    df1 = pd.read_csv(TRAINING_CONFIG['train_csv'])
    df2 = pd.read_csv(TRAINING_CONFIG['test_csv'])
    
    # Merge OCR data
    try:
        df_train_ocr = pd.read_csv('/kaggle/input/results-ocr/cuet_cse_train_text_hunyuanocr.csv')
        df_test_ocr = pd.read_csv('/kaggle/input/results-ocr/cuet_cse_test_text_hunyuanocr.csv')
        df_train = pd.merge(df1, df_train_ocr, on='Image_name', how='left')
        df_test = pd.merge(df2, df_test_ocr, on='Image_name', how='left')
        print("✓ OCR data merged successfully")
    except FileNotFoundError:
        print("⚠ OCR files not found, proceeding without OCR merge.")
        df_train = df1
        df_test = df2
    
    # Merge context data
    df_context_train = pd.read_csv(TRAINING_CONFIG['train_context_csv'])
    df_train = pd.merge(df_train, df_context_train, on='Image_name', how='left', suffixes=('', '_ctx'))
    
    if TRAINING_CONFIG['test_context_csv'] and os.path.exists(TRAINING_CONFIG['test_context_csv']):
        df_context_test = pd.read_csv(TRAINING_CONFIG['test_context_csv'])
        df_test = pd.merge(df_test, df_context_test, on='Image_name', how='left', suffixes=('', '_ctx'))
        print("✓ Test context loaded successfully")
    else:
        print("⚠ Test context not available - using empty strings")
        for col in ['combined_meaning', 'context_analysis']:
            if col not in df_test.columns:
                df_test[col] = ''
    
    # Fill NaN values
    text_cols = ['text', 'combined_meaning', 'context_analysis']
    for df in [df_train, df_test]:
        for col in text_cols:
            if col in df.columns:
                df[col] = df[col].fillna('').astype(str)
            else:
                df[col] = ''

    print(f"✓ Train shape: {df_train.shape}")
    print(f"✓ Test shape: {df_test.shape}")
    
    return df_train, df_test

# ==========================================
# DUAL-ENCODER TEXT CLASSIFIER
# ==========================================
class DualEncoderTextClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        
        # ========== PRIMARY ENCODER (DeBERTa-v3-Large) ==========
        self.primary_encoder = AutoModel.from_pretrained(TRAINING_CONFIG['primary_model'])
        primary_dim = self.primary_encoder.config.hidden_size
        
        # ========== SECONDARY ENCODER (RoBERTa-Large) ==========
        if TRAINING_CONFIG['use_dual_encoder']:
            self.secondary_encoder = AutoModel.from_pretrained(TRAINING_CONFIG['secondary_model'])
            secondary_dim = self.secondary_encoder.config.hidden_size
        else:
            self.secondary_encoder = None
            secondary_dim = 0
        
        # Enable gradient checkpointing if configured
        if TRAINING_CONFIG['use_gradient_checkpointing']:
            self.primary_encoder.gradient_checkpointing_enable()
            if self.secondary_encoder:
                self.secondary_encoder.gradient_checkpointing_enable()
        
        # ========== OCR TEXT PROCESSING ==========
        self.ocr_projector = nn.Sequential(
            nn.Linear(primary_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # ========== CONTEXT PROCESSING (Combined Meaning) ==========
        self.context_combined_projector = nn.Sequential(
            nn.Linear(primary_dim, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(768, 512),
            nn.LayerNorm(512),
            nn.GELU()
        )
        
        # ========== CONTEXT PROCESSING (Context Analysis) ==========
        if TRAINING_CONFIG['use_dual_encoder']:
            self.context_analysis_projector = nn.Sequential(
                nn.Linear(secondary_dim, 768),
                nn.LayerNorm(768),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(768, 512),
                nn.LayerNorm(512),
                nn.GELU()
            )
        else:
            self.context_analysis_projector = nn.Sequential(
                nn.Linear(primary_dim, 768),
                nn.LayerNorm(768),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(768, 512),
                nn.LayerNorm(512),
                nn.GELU()
            )
        
        # ========== CROSS-CONTEXT ATTENTION ==========
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=512,
            num_heads=8,
            dropout=0.2,
            batch_first=True
        )
        
        # ========== FEATURE FUSION ==========
        # OCR (512) + Combined (512) + Analysis (512) = 1536
        fusion_dim = 512 + 512 + 512
        
        self.fusion_gate = nn.Sequential(
            nn.Linear(fusion_dim, fusion_dim),
            nn.Sigmoid()
        )
        
        # ========== CLASSIFIER HEAD ==========
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.5),
            
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.4),
            
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, num_classes)
        )
        
    def forward(self, ocr_input, combined_input, analysis_input):
        # ========== OCR TEXT ENCODING ==========
        ocr_output = self.primary_encoder(
            input_ids=ocr_input['ids'], 
            attention_mask=ocr_input['mask']
        )
        ocr_emb = ocr_output.last_hidden_state[:, 0, :]  # CLS token
        ocr_feat = self.ocr_projector(ocr_emb) * TRAINING_CONFIG['ocr_weight']
        
        # ========== COMBINED MEANING ENCODING ==========
        combined_output = self.primary_encoder(
            input_ids=combined_input['ids'],
            attention_mask=combined_input['mask']
        )
        combined_emb = combined_output.last_hidden_state[:, 0, :]
        combined_feat = self.context_combined_projector(combined_emb) * TRAINING_CONFIG['context_weight']
        
        # ========== CONTEXT ANALYSIS ENCODING ==========
        if TRAINING_CONFIG['use_dual_encoder']:
            analysis_output = self.secondary_encoder(
                input_ids=analysis_input['ids'],
                attention_mask=analysis_input['mask']
            )
        else:
            analysis_output = self.primary_encoder(
                input_ids=analysis_input['ids'],
                attention_mask=analysis_input['mask']
            )
        
        analysis_emb = analysis_output.last_hidden_state[:, 0, :]
        analysis_feat = self.context_analysis_projector(analysis_emb) * TRAINING_CONFIG['context_weight']
        
        # ========== CROSS-ATTENTION BETWEEN CONTEXTS ==========
        combined_attn = combined_feat.unsqueeze(1)
        analysis_attn = analysis_feat.unsqueeze(1)
        
        attended_combined, _ = self.cross_attention(combined_attn, analysis_attn, analysis_attn)
        attended_combined = attended_combined.squeeze(1)
        
        attended_analysis, _ = self.cross_attention(analysis_attn, combined_attn, combined_attn)
        attended_analysis = attended_analysis.squeeze(1)
        
        # ========== FEATURE FUSION ==========
        all_features = torch.cat([ocr_feat, attended_combined, attended_analysis], dim=1)
        
        gate = self.fusion_gate(all_features)
        gated_features = all_features * gate
        
        # ========== CLASSIFICATION ==========
        logits = self.classifier(gated_features)
        
        return logits

# ==========================================
# DATASET
# ==========================================
class TextOnlyDataset(Dataset):
    def __init__(self, df, primary_tokenizer, secondary_tokenizer, is_train=True):
        self.df = df.reset_index(drop=True)
        self.primary_tokenizer = primary_tokenizer
        self.secondary_tokenizer = secondary_tokenizer
        self.is_train = is_train
        
        if self.is_train:
            self.labels = [LABEL2ID[x] for x in df['Label']]

    def __len__(self): 
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Tokenize OCR text with primary tokenizer
        ocr_text = str(row['text']) if len(str(row['text'])) > 1 else "unclear text"
        ocr_tok = self.primary_tokenizer(
            ocr_text,
            padding='max_length',
            truncation=True,
            max_length=TRAINING_CONFIG['max_len_secondary'],
            return_tensors='pt'
        )
        
        # Tokenize combined meaning with primary tokenizer
        combined_tok = self.primary_tokenizer(
            str(row['combined_meaning']),
            padding='max_length',
            truncation=True,
            max_length=TRAINING_CONFIG['max_len_primary'],
            return_tensors='pt'
        )
        
        # Tokenize context analysis with secondary tokenizer (if dual encoder)
        if TRAINING_CONFIG['use_dual_encoder']:
            analysis_tok = self.secondary_tokenizer(
                str(row['context_analysis']),
                padding='max_length',
                truncation=True,
                max_length=TRAINING_CONFIG['max_len_primary'],
                return_tensors='pt'
            )
        else:
            analysis_tok = self.primary_tokenizer(
                str(row['context_analysis']),
                padding='max_length',
                truncation=True,
                max_length=TRAINING_CONFIG['max_len_primary'],
                return_tensors='pt'
            )
        
        item = {
            'ocr_ids': ocr_tok['input_ids'].squeeze(0),
            'ocr_mask': ocr_tok['attention_mask'].squeeze(0),
            'combined_ids': combined_tok['input_ids'].squeeze(0),
            'combined_mask': combined_tok['attention_mask'].squeeze(0),
            'analysis_ids': analysis_tok['input_ids'].squeeze(0),
            'analysis_mask': analysis_tok['attention_mask'].squeeze(0),
        }
        
        if self.is_train:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            
        return item

# ==========================================
# TRAINING LOOP
# ==========================================
def train_kfold():
    print("="*50)
    print("DUAL-ENCODER TEXT-ONLY CLASSIFICATION")
    print(f"Primary Model: {TRAINING_CONFIG['primary_model']}")
    print(f"Secondary Model: {TRAINING_CONFIG['secondary_model']}")
    print(f"Batch Size: {TRAINING_CONFIG['batch_size']}")
    print(f"Gradient Accumulation: {TRAINING_CONFIG['gradient_accumulation_steps']}")
    print(f"Effective Batch Size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
    print("="*50)
    
    df_train, df_test = load_and_prepare_data()
    
    primary_tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['primary_model'])
    secondary_tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['secondary_model']) if TRAINING_CONFIG['use_dual_encoder'] else primary_tokenizer
    
    skf = StratifiedKFold(n_splits=TRAINING_CONFIG['num_folds'], shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['Label'])):
        print(f"\n{'='*40}")
        print(f"FOLD {fold+1}/{TRAINING_CONFIG['num_folds']}")
        print(f"{'='*40}")

        gc.collect()
        torch.cuda.empty_cache()
        
        scaler = torch.cuda.amp.GradScaler() if TRAINING_CONFIG['mixed_precision'] else None
        
        d_train = TextOnlyDataset(df_train.iloc[train_idx], primary_tokenizer, secondary_tokenizer)
        d_val = TextOnlyDataset(df_train.iloc[val_idx], primary_tokenizer, secondary_tokenizer)
        
        l_train = DataLoader(d_train, batch_size=TRAINING_CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
        l_val = DataLoader(d_val, batch_size=TRAINING_CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
        
        model = DualEncoderTextClassifier(num_classes=TRAINING_CONFIG['num_classes']).to(TRAINING_CONFIG['device'])
        
        optimizer = AdamW(model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'])
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=len(l_train) // TRAINING_CONFIG['gradient_accumulation_steps'], T_mult=1)
        criterion = nn.CrossEntropyLoss()
        
        best_macro_f1 = 0
        patience = 0
        
        for epoch in range(TRAINING_CONFIG['epochs']):
            model.train()
            train_loss = 0
            optimizer.zero_grad()
            
            pbar = tqdm(l_train, desc=f"Epoch {epoch+1}")
            for step, batch in enumerate(pbar):
                labels = batch['labels'].to(TRAINING_CONFIG['device'])
                
                ocr_input = {
                    'ids': batch['ocr_ids'].to(TRAINING_CONFIG['device']),
                    'mask': batch['ocr_mask'].to(TRAINING_CONFIG['device'])
                }
                
                combined_input = {
                    'ids': batch['combined_ids'].to(TRAINING_CONFIG['device']),
                    'mask': batch['combined_mask'].to(TRAINING_CONFIG['device'])
                }
                
                analysis_input = {
                    'ids': batch['analysis_ids'].to(TRAINING_CONFIG['device']),
                    'mask': batch['analysis_mask'].to(TRAINING_CONFIG['device'])
                }
                
                is_update_step = (step + 1) % TRAINING_CONFIG['gradient_accumulation_steps'] == 0
                
                if TRAINING_CONFIG['mixed_precision']:
                    with torch.cuda.amp.autocast():
                        logits = model(ocr_input, combined_input, analysis_input)
                        loss = criterion(logits, labels)
                        loss = loss / TRAINING_CONFIG['gradient_accumulation_steps']
                    
                    scaler.scale(loss).backward()
                else:
                    logits = model(ocr_input, combined_input, analysis_input)
                    loss = criterion(logits, labels)
                    loss = loss / TRAINING_CONFIG['gradient_accumulation_steps']
                    loss.backward()
                
                if is_update_step:
                    if TRAINING_CONFIG['mixed_precision']:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                    
                    scheduler.step()
                    optimizer.zero_grad()
                
                if step % TRAINING_CONFIG['empty_cache_freq'] == 0:
                    torch.cuda.empty_cache()
                
                train_loss += loss.item()
                pbar.set_postfix({'loss': f"{train_loss * TRAINING_CONFIG['gradient_accumulation_steps'] / (step + 1):.4f}"})
            
            # Validation
            model.eval()
            preds, trues = [], []
            
            with torch.no_grad():
                for batch in tqdm(l_val, desc="Validating", leave=False):
                    labels = batch['labels'].to(TRAINING_CONFIG['device'])
                    
                    ocr_input = {
                        'ids': batch['ocr_ids'].to(TRAINING_CONFIG['device']),
                        'mask': batch['ocr_mask'].to(TRAINING_CONFIG['device'])
                    }
                    
                    combined_input = {
                        'ids': batch['combined_ids'].to(TRAINING_CONFIG['device']),
                        'mask': batch['combined_mask'].to(TRAINING_CONFIG['device'])
                    }
                    
                    analysis_input = {
                        'ids': batch['analysis_ids'].to(TRAINING_CONFIG['device']),
                        'mask': batch['analysis_mask'].to(TRAINING_CONFIG['device'])
                    }
                    
                    if TRAINING_CONFIG['mixed_precision']:
                        with torch.cuda.amp.autocast():
                            logits = model(ocr_input, combined_input, analysis_input)
                    else:
                        logits = model(ocr_input, combined_input, analysis_input)
                    
                    preds.extend(torch.argmax(logits, 1).cpu().numpy())
                    trues.extend(labels.cpu().numpy())
            
            macro_f1 = f1_score(trues, preds, average='macro')
            print(f"Epoch {epoch+1}: Macro F1={macro_f1:.4f}")
            
            if macro_f1 > best_macro_f1:
                best_macro_f1 = macro_f1
                torch.save(model.state_dict(), f"{TRAINING_CONFIG['output_dir']}/text_classifier_fold{fold}.pth")
                patience = 0
                print(f"✓ Best model saved (F1: {best_macro_f1:.4f})")
            else:
                patience += 1
                if patience >= 3:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
        
        fold_scores.append(best_macro_f1)
        del model, optimizer
        if scaler: del scaler
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\n{'='*50}")
    print(f"TRAINING COMPLETE")
    print(f"Average Macro F1: {np.mean(fold_scores):.4f}")
    print(f"Fold Scores: {fold_scores}")
    print(f"{'='*50}")

# ==========================================
# INFERENCE
# ==========================================
def generate_submission():
    print("Generating Submission...")
    _, df_test = load_and_prepare_data()
    
    primary_tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['primary_model'])
    secondary_tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['secondary_model']) if TRAINING_CONFIG['use_dual_encoder'] else primary_tokenizer
    
    test_dataset = TextOnlyDataset(df_test, primary_tokenizer, secondary_tokenizer, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    avg_probs = np.zeros((len(df_test), TRAINING_CONFIG['num_classes']))
    
    for fold in range(TRAINING_CONFIG['num_folds']):
        path = f"{TRAINING_CONFIG['output_dir']}/text_classifier_fold{fold}.pth"
        if not os.path.exists(path): 
            print(f"⚠ Fold {fold} model not found")
            continue
            
        model = DualEncoderTextClassifier(num_classes=TRAINING_CONFIG['num_classes']).to(TRAINING_CONFIG['device'])
        model.load_state_dict(torch.load(path))
        model.eval()
        
        fold_probs = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Fold {fold}"):
                ocr_input = {
                    'ids': batch['ocr_ids'].to(TRAINING_CONFIG['device']),
                    'mask': batch['ocr_mask'].to(TRAINING_CONFIG['device'])
                }
                
                combined_input = {
                    'ids': batch['combined_ids'].to(TRAINING_CONFIG['device']),
                    'mask': batch['combined_mask'].to(TRAINING_CONFIG['device'])
                }
                
                analysis_input = {
                    'ids': batch['analysis_ids'].to(TRAINING_CONFIG['device']),
                    'mask': batch['analysis_mask'].to(TRAINING_CONFIG['device'])
                }
                
                if TRAINING_CONFIG['mixed_precision']:
                    with torch.cuda.amp.autocast():
                        logits = model(ocr_input, combined_input, analysis_input)
                else:
                    logits = model(ocr_input, combined_input, analysis_input)
                    
                fold_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        
        avg_probs += np.array(fold_probs)
        
        del model
        torch.cuda.empty_cache()
    
    avg_probs /= TRAINING_CONFIG['num_folds']
    submission = pd.DataFrame({
        'Image_name': df_test['Image_name'], 
        'Target': [ID2LABEL[p] for p in np.argmax(avg_probs, axis=1)]
    })
    submission.to_csv(f"{TRAINING_CONFIG['output_dir']}/submission only text proecssing.csv", index=False)
    print("✓ Submission saved.")

if __name__ == "__main__":
    os.makedirs(TRAINING_CONFIG['output_dir'], exist_ok=True)
    train_kfold()
    generate_submission()

DUAL-ENCODER TEXT-ONLY CLASSIFICATION
Primary Model: microsoft/deberta-v3-base
Secondary Model: roberta-base
Batch Size: 4
Gradient Accumulation: 8
Effective Batch Size: 32
✓ OCR data merged successfully
✓ Test context loaded successfully
✓ Train shape: (2860, 8)
✓ Test shape: (330, 6)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Epoch 9: Macro F1=0.9083
✓ Best model saved (F1: 0.9083)


Epoch 10: 100%|██████████| 572/572 [02:28<00:00,  3.84it/s, loss=0.0902]


Epoch 10: Macro F1=0.9159
✓ Best model saved (F1: 0.9159)

TRAINING COMPLETE
Average Macro F1: 0.9262
Fold Scores: [0.932811605268181, 0.9312957927954488, 0.9216331004247158, 0.9296026255431034, 0.9158769604386405]
Generating Submission...
✓ OCR data merged successfully
✓ Test context loaded successfully
✓ Train shape: (2860, 8)
✓ Test shape: (330, 6)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 0: 100%|██████████| 42/42 [00:05<00:00,  7.24it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 1: 100%|██████████| 42/42 [00:05<00:00,  7.30it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 2: 100%|██████████| 42/42 [00:05<00:00,  7.24it/s]
Some we

✓ Submission saved.
